In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pip

##  -U is for upgrade
##  -qqq is for quiet

!pip install -U git+https://github.com/qdosgit4/30040-experiments.git#subdirectory=experiment_5_reparameterisation_reimplementation

##  !pip show -f reparam

  Cloning https://github.com/qdosgit4/30040-experiments.git to /tmp/pip-req-build-v2v04kc_
  Running command git clone --filter=blob:none --quiet https://github.com/qdosgit4/30040-experiments.git /tmp/pip-req-build-v2v04kc_
  Resolved https://github.com/qdosgit4/30040-experiments.git to commit 33c9b6a4ffc2ebeba1415a6d19363052c54fdd0d


In [3]:
import argparse, time
from datetime import datetime

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset, Subset
from torchvision import datasets
from torchvision.transforms import ToTensor

import matplotlib.pyplot as plt

from reparam.linear_bayesian import Linear_bayesian
from reparam.mini_model_reparam import Linear_model
from reparam.pi_dataset import Pi_dataset
from reparam.experiment_training_lib import run_training_loop
##  from reparam.experiment_utilise_lib import run_utilisation_loop_once, run_utilisation_loop_batch

print(torch.__version__)

2.10.0+cu128


In [4]:
##import pkgutil

##for module in pkgutil.iter_modules():
##  print(module.name)

In [5]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"

##  Following is essential for reparameterisation;
##    bfloat16 can cause vanishing gradient.
torch.set_default_dtype(torch.float32)

In [6]:
def run_model(env):

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    ##  Setup filename to store params to.

    filename = f"model_params_batch_{env['batch_n']}_epochs_{env['training_epochs']}_{env['params_name']}"


    ##  Define model.

    model = Linear_model(
                env["neurons_n"],
                env["mu_w_init"],
                env["mu_b_init"],
                env["rho_w_init"],
                env["rho_b_init"],
    ).to(device)

    ##  Ensure multiple GPUs used if available.

    if torch.cuda.device_count() > 1:

        print(f"Using {torch.cuda.device_count()} GPUs!")

        model = nn.DataParallel(model, device_ids=[0, 1])

    ##  Either load params or load training data.

    print(env)

    if env["params_load"] is not None:

        run_utilisation_loop_once(
                    model,
                    env["params_load"],
                    env["params_load"].split('/')[-1] + "_plot",
                    env["sample_quantity"]
        )

    elif env["params_load_batch"] is not None:

        run_utilisation_loop_batch(
                    model,
                    env["params_load_batch"],
                    env["sample_quantity"]
        )

    else:

        ##  Load up limited training data to induce epistemic uncertainty.

        train_dl = DataLoader(
                    Subset(
                        Pi_dataset("/content/drive/MyDrive/modules_work/COMP30040/datasets/pi_dataset_40000.txt"),
                        range(40000)
                    ),
                    batch_size = env["batch_n"],
                    shuffle=True
        )

        test_dl = DataLoader(
                    Subset(
                        Pi_dataset("/content/drive/MyDrive/modules_work/COMP30040/datasets/pi_dataset_8000.txt"),
                        range(8000)
                    ),
                    batch_size = env["batch_n"],
                    shuffle=True
        )

        run_training_loop(model,
                          train_dl,
                          test_dl,
                          env["training_epochs"],
                          filename,
                          env["kl_ratio"])


In [1]:
env_train = {
            "neurons_n": 1024,
            "batch_n": 64,
            "training_epochs": 10,
            "params_name": "test",      ##  Filename to store params to.
            "params_load": None,        ##  Filename to load specific set of params from.
            "params_load_batch": None,  ##  Filename to wildcard match a multiple sets of params.
            "random_seed": 239852,
            "mu_w_init": 0.1,
            "mu_b_init": -3.0,
            "rho_w_init": 0.1,
            "rho_b_init": -3.0,
            "kl_loss_ratio": 0.0,       ##  Ratio to consider KL-divergence loss during sum of loss functions.
}

#for k, v in env.items():
#            print(f"{k}: {v}")

torch.manual_seed(env_train["random_seed"])

run_model(env_train)

NameError: name 'torch' is not defined

In [8]:
#!/usr/bin/env python3

from pathlib import Path
import lzma
import io

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset, Subset
from torchvision import datasets
from torchvision.transforms import ToTensor

# import matplotlib.pyplot as plt


torch.set_default_dtype(torch.float32)

device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"


def run_utilisation_loop_batch(model: nn.Module, batch_name: str, sample_quantity: int):

    print("loop batch")

    directory = Path('./params/')

    # Find all files with batch_name in filename

    for file in directory.glob(f"model_params*{batch_name}*xz"):

        print(str(file))

        if file.is_file():

            dist_metadata = str(file).split(batch_name)[-1].replace('.xz','')

            graph_filename = f"{batch_name}{dist_metadata}"

            params_path = f"./{str(file)}"

            run_utilisation_loop_once(model, params_path, graph_filename)

            # Do something with the file


def run_utilisation_loop_once(model: nn.Module, params_path: str, graph_filename: str, sample_quantity: int):

    ##  Decompress to memory.

    # with lzma.open(params_path, 'rb') as f:

    #     decompressed_data = f.read()

    ##  Load params.

    # checkpoint = torch.load(io.BytesIO(decompressed_data),

    checkpoint = torch.load(params_path,
                           params_only = True,
                           ##map_location=torch.device('cpu')
                           )

    old_state_dict = checkpoint['state_dict'] if 'state_dict' in checkpoint else checkpoint

    print(old_state_dict)

    ##  Fix naming scheme and load module.

    model.load_state_dict(
                {k.replace('module.', '', 1): v for k, v in old_state_dict.items()}
    )

    res = []

    interval = 1 / (2 ** 6)

    dl = DataLoader(
                TensorDataset(
                    torch.arange(1.57, 4.71 + interval, interval)
                ),
                batch_size = 1,
                shuffle = False
    )

    with torch.no_grad():

        # all_tensors = []

        # for i in range(1):

            outputs_set = ()

            for batch, (X,) in enumerate(dl):

                X = X.to(device)

                # print(X)

                ##  Run X through model, generate prediction.

                sample_set = ()

                for _ in range(sample_quantity):

                    y_hat_sample, _ = model(X)

                    sample_set = sample_set + (y_hat_sample,)

                print(sample_set)

                ##  Take mean of samples.

                y_hat = torch.stack(sample_set).mean(dim=0)

                print(f"p({X.item()}) = {y_hat.item()}")

                ##  Store for later plotting

                outputs_set = outputs_set + ((X.item(), y_hat.squeeze(0).item()),)

            # inner_tensor = torch.stack(outputs_set, dim=0)

            # all_tensors.append(inner_tensor)

            # print(all_tensors)

            x, y = zip(*outputs_set)

            plt.figure(figsize=(6, 4))

            plt.plot(
                        x, y,
                        marker='o',
                        markersize=4,          # small dots
                        markerfacecolor='black',
                        markeredgecolor='black',
                        linewidth=0.8,         # thin connecting line
                        color='black',
                        label='Trajectory'
            )

            # plt.scatter(
            #             x, y,
            #             color='black',          # point colour
            #             edgecolor='black',          # optional border around each marker
            #             s=10,                       # marker size
            #             label='Samples'
            # )

            plt.xlabel('Input value')
            plt.ylabel('Probability of π classification')
            plt.grid(True, ls='--', lw=0.5, alpha=0.7)
            plt.tight_layout()

            plt.tight_layout()

            plt.savefig(f"{graph_filename}.pdf", dpi=300)
            plt.close()

            return graph_filename



In [10]:
env_load = {
            "params_name": None,
            "params_load": "model_params_batch_64_epochs_10_test",
            "params_load_batch": None,
            "sample_quantity": 10
}

env_utilise = env_train | env_load

for k, v in env_utilise.items():
            print(f"{k}: {v}")

run_model(env_utilise)

Streaming output truncated to the last 5000 lines.
Sigma summation: tensor(761.3845214844, device='cuda:0')
Sigma summation: tensor(780556.3750000000, device='cuda:0')
Sigma summation: tensor(762.3653564453, device='cuda:0')
Sigma summation: tensor(761.3845214844, device='cuda:0')
Sigma summation: tensor(780556.3750000000, device='cuda:0')
Sigma summation: tensor(762.3653564453, device='cuda:0')
(tensor([1.], device='cuda:0'), tensor([1.], device='cuda:0'), tensor([1.], device='cuda:0'), tensor([1.], device='cuda:0'), tensor([1.], device='cuda:0'), tensor([1.], device='cuda:0'), tensor([1.], device='cuda:0'), tensor([1.], device='cuda:0'), tensor([1.], device='cuda:0'), tensor([1.], device='cuda:0'))
p(2.273124933242798) = 1.0
Sigma summation: tensor(761.3845214844, device='cuda:0')
Sigma summation: tensor(780556.3750000000, device='cuda:0')
Sigma summation: tensor(762.3653564453, device='cuda:0')
Sigma summation: tensor(761.3845214844, device='cuda:0')
Sigma summation: tensor(780556.3